In [35]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [36]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [273]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [312]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model(
    tokenizer=TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer"), 
    device="cuda", depth=6, head_dim=128, context_size=4096, nb_heads_mult=1.0)
model.show_infos()

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
1_589_356 total params (embeding: 327_680 | last layer: 81_920 | transformer: 1_179_744)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [313]:
dataset = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=model.context_size,
    tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)

In [314]:
import importlib
import metrics.metrics
_ = importlib.reload(metrics.metrics)
import metrics.metrics

In [ ]:
N = 8
testDevice = model.device # torch.device("cuda:0")

prof = Profiler(["iterDataloader", "split", "target", "logits", "print"])

dataloader = DataLoader(dataset, batch_size=N, shuffle=True, num_workers=0)
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)


print(f"there are {len(dataloader)} batch to process")
it = iter(enumerate(dataloader))
i = 0
t0 = time.perf_counter()
while True:
    with prof.mesure("iterDataloader"):
        try: i, datas = next(it)
        except StopIteration:
            break
    
    with prof.mesure("print"):
        if i % 5 == 0:
            print(f"\rdoing: {i=}", end="", flush=True)
            
    with prof.mesure("split"):
        tokens: torch.Tensor = datas["tokens"].to(torch.int64)
        assert isinstance(tokens, torch.Tensor)
        dtIndexes: list[int] = datas["datasetIndex"].tolist()
        svgIndex: list[int] = datas["svgIndex"].tolist()
        chunckIndex: list[int] = datas["chunckIndex"].tolist()
        nbChars: int = sum([len(dataset.chunks[i].text) for i in dtIndexes])

    with prof.mesure("target"):
        if True:
            target = tokens
        else: target = torch.tensor(list(range(256)), dtype=torch.int64).view(1, -1)
        target = target.to(testDevice)

    with prof.mesure("logits"):
        batchSize, seq_len = target.shape
        vocab_size = model.tokenizer.get_vocab_size()
        if True:
            logits = torch.full((batchSize, seq_len, vocab_size), -10.0, device=testDevice)
            mask = (target != svg_dataset.IGNORE_INDEX)
            b, t = mask.nonzero(as_tuple=True)
            logits[b, t, target[b, t]] = 0.0
            logits += 3.0 * torch.randn(batchSize, seq_len, vocab_size, device=testDevice)
        else: logits = torch.randn(batchSize, seq_len, vocab_size, device=testDevice)# * 0.001

    # print(f"logits of shape: {logits.shape}[{logits.dtype}]")
    # print(f"target of shape: {target.shape}[{target.dtype}]")
    
    accum.batch_logits_metrics(logits, target, totalNbChars=nbChars)
    
    #if i*N > 50000:
    #    break

torch.cuda.empty_cache()
t1 = time.perf_counter()
print()
print(f"performed: {i} batch ({i*N} chuncks) in {prettyTime(t1-t0)}")
print(f" -> {i/(t1-t0):.2f} batch/sec | {i*N/(t1-t0):.2f} chuncks/sec")

print()
res = accum.get_metrics()
prettyPrint(prof.pretty_totalTimes())
prettyPrint(accum._prof.pretty_totalTimes())

there are 1725 batch to process
doing: i=1720
performed: 1724 batch (13792 chuncks) in 21.319 sec
 -> 80.87 batch/sec | 646.93 chuncks/sec

{
    iterDataloader: 317.566 ms,
    split: 103.839 ms,
    target: 243.682 ms,
    logits: 2.229 sec,
    print: 195.616 ms
}
{
    loss1: 177.688 ms,
    loss2: 101.631 ms,
    filter: 3.508 sec,
    CE_related: 1.161 sec,
    entropy: 4.238 sec,
    SD: 582.907 ms,
    topK: 333.555 ms,
    accuacy: 7.924 sec
}


In [308]:
prettyPrint(res)

{
    CE_train: 1.7257725038035228,
    CE2_train: 0.7904048350267059,
    PPL_train: 5.616858398637252,
    PPL2_train: 2.204288618890233,
    BPC_train: 1.2613431136163986,
    ENTROPY_train: 2.0384026365884904,
    LOGITS_SD_train: 3.0270588629355952,
    TOP-1_train: 0.5947900582246645,
    TOP-5_train: 0.8167903747336008
}


In [315]:
metrics.metrics.get_learning_rates(model)

{'lr_lm_head': 0.009797958971132713,
 'lr_embedding': 0.4898979485566357,
 'lr_value_embeds': 0.4898979485566357,
 'lr_residuals': 0.005,
 'lr_x0': 0.5,
 'lr_transformers_grp_0': 0.02,
 'lr_transformers_grp_1': 0.02,
 'lr_transformers_grp_2': 0.02,
 'lr_transformers_grp_3': 0.02}

In [329]:
N = 16
testDevice = torch.device("cuda:0")

prof = Profiler([
    "print", "iterDataloader", "split", "tokens",
    "forward", "metrics&loss", "zero_grad", "backward", "step"])

dataloader = DataLoader(dataset, batch_size=N, shuffle=True, num_workers=0)
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)


print(f"there are {len(dataloader)} batch to process")
it = iter(enumerate(dataloader, start=1))
i = 0
t0 = time.perf_counter()
while True:
    torch.cuda.empty_cache()
    with prof.mesure("iterDataloader"):
        try: i, datas = next(it)
        except StopIteration:
            break
    
    with prof.mesure("print"):
        if i % 5 == 0:
            print(f"\rdoing: {i=}", end="", flush=True)
            
    with prof.mesure("split"):
        tokens: torch.Tensor = datas["tokens"].to(torch.int64)
        assert isinstance(tokens, torch.Tensor)
        dtIndexes: list[int] = datas["datasetIndex"].tolist()
        svgIndex: list[int] = datas["svgIndex"].tolist()
        chunckIndex: list[int] = datas["chunckIndex"].tolist()
        nbChars: int = sum([len(dataset.chunks[i].text) for i in dtIndexes])

    with prof.mesure("tokens"):
        tokens = tokens.to(testDevice)

    with prof.mesure("forward"):
        logits = model.llm.forward(idx=tokens, targets=None)
        
    # print(f"tokens of shape: {tokens.shape}[{tokens.dtype}]")
    # print(f"logits of shape: {logits.shape}[{logits.dtype}]")
    
    with prof.mesure("metrics&loss"):
        loss = accum.batch_logits_metrics(
            logits[:, : -1].contiguous(), 
            tokens[:, 1:].contiguous(),
            totalNbChars=nbChars)
    
    with prof.mesure("zero_grad"):
        model.optimizer.zero_grad()
    with prof.mesure("backward"):
        loss.backward()
    with prof.mesure("step"):
        model.optimizer.step()
    
    if i*N >= 5000:
        break

torch.cuda.empty_cache()
t1 = time.perf_counter()
print()
print(f"performed: {i} batch ({i*N} chuncks) in {prettyTime(t1-t0)}")
print(f" -> {i/(t1-t0):.2f} batch/sec | {i*N/(t1-t0):.2f} chuncks/sec")
res = accum.get_metrics()
prettyPrint(prof.pretty_totalTimes())
prettyPrint(accum._prof.pretty_totalTimes())
prettyPrint(res)


there are 863 batch to process
doing: i=310
performed: 313 batch (5008 chuncks) in 40.187 sec
 -> 7.79 batch/sec | 124.62 chuncks/sec
{
    print: 41.874 ms,
    iterDataloader: 100.480 ms,
    split: 63.623 ms,
    tokens: 47.708 ms,
    forward: 12.069 sec,
    metrics&loss: 9.903 sec,
    zero_grad: 42.575 ms,
    backward: 1.471 sec,
    step: 1.460 sec
}
{
    loss1: 412.287 ms,
    loss2: 356.238 ms,
    filter: 4.082 sec,
    CE_related: 1.135 sec,
    entropy: 2.610 sec,
    SD: 130.137 ms,
    topK: 147.209 ms,
    accuacy: 969.108 ms
}
{
    CE_train: 0.8930341408774287,
    CE2_train: 0.6866708397111452,
    PPL_train: 2.4425293979667844,
    PPL2_train: 1.9870891708938996,
    BPC_train: 0.65258506765451,
    ENTROPY_train: 0.8927582764266281,
    LOGITS_SD_train: 3.7768282275317335,
    TOP-1_train: 0.7771307067027459,
    TOP-5_train: 0.8483144546256213
}


In [328]:
prettyPrint(res)

{
    CE_train: 0.9847874440726623,
    CE2_train: 0.6929305432299466,
    PPL_train: 2.6772427601202353,
    PPL2_train: 1.9995667722683466,
    BPC_train: 0.7197512624075658,
    ENTROPY_train: 0.9833142291926158,
    LOGITS_SD_train: 3.7142189257658424,
    TOP-1_train: 0.7505734796219333,
    TOP-5_train: 0.8385858862555601
}


In [330]:
prettyPrint(res)

{
    CE_train: 0.8930341408774287,
    CE2_train: 0.6866708397111452,
    PPL_train: 2.4425293979667844,
    PPL2_train: 1.9870891708938996,
    BPC_train: 0.65258506765451,
    ENTROPY_train: 0.8927582764266281,
    LOGITS_SD_train: 3.7768282275317335,
    TOP-1_train: 0.7771307067027459,
    TOP-5_train: 0.8483144546256213
}


# stats when testing

there are 13793 batch to process
doing: i=500
performed: 500 batch (500 chuncks) in 18.993 sec
 -> 26.38 batch/sec | 26.38 chuncks/sec
{
    print: 69.003 ms,
    iterDataloader: 100.535 ms,
    split: 28.354 ms,
    tokens: 86.609 ms,
    forward: 7.559 sec,
    metrics&loss: 2.347 sec,
    zero_grad: 57.810 ms,
    backward: 2.479 sec,
    step: 2.458 sec
}

there are 6897 batch to process
doing: i=250
performed: 250 batch (500 chuncks) in 9.353 sec
 -> 26.73 batch/sec | 53.46 chuncks/sec
{
    print: 37.736 ms,
    iterDataloader: 58.378 ms,
    split: 15.162 ms,
    tokens: 41.625 ms,
    forward: 1.953 sec,
    metrics&loss: 3.115 sec,
    zero_grad: 31.538 ms,
    backward: 1.206 sec,
    step: 1.501 sec
}

there are 3449 batch to process
doing: i=125
performed: 125 batch (500 chuncks) in 7.436 sec
 -> 16.81 batch/sec | 67.24 chuncks/sec
{
    print: 18.826 ms,
    iterDataloader: 35.273 ms,
    split: 7.954 ms,
    tokens: 15.757 ms,
    forward: 1.688 sec,
    metrics&loss: 2.804 sec,
    zero_grad: 18.196 ms,
    backward: 603.035 ms,
    step: 697.346 ms
}

there are 1725 batch to process
doing: i=60
performed: 63 batch (504 chuncks) in 6.883 sec
 -> 9.15 batch/sec | 73.22 chuncks/sec
{
    print: 11.293 ms,
    iterDataloader: 28.092 ms,
    split: 6.280 ms,
    tokens: 9.228 ms,
    forward: 1.543 sec,
    metrics&loss: 2.764 sec,
    zero_grad: 11.558 ms,
    backward: 302.809 ms,
    step: 381.092 ms
}

there are 863 batch to process
doing: i=30
performed: 32 batch (512 chuncks) in 6.476 sec
 -> 4.94 batch/sec | 79.07 chuncks/sec
{
    print: 5.502 ms,
    iterDataloader: 18.805 ms,
    split: 7.956 ms,
    tokens: 5.724 ms,
    forward: 1.413 sec,
    metrics&loss: 2.599 sec,
    zero_grad: 7.059 ms,
    backward: 164.066 ms,
    step: 168.861 ms
}

there are 863 batch to process
doing: i=30
performed: 32 batch (512 chuncks) in 4.626 sec
 -> 6.92 batch/sec | 110.68 chuncks/sec
{
    print: 4.346 ms,
    iterDataloader: 12.407 ms,
    split: 6.354 ms,
    tokens: 5.839 ms,
    forward: 1.270 sec,
    metrics&loss: 3.479 ms,
    zero_grad: 3.922 ms,
    backward: 354.835 ms,
    step: 148.873 ms
}